<a href="https://colab.research.google.com/github/GopalKrishna-India/Geospatial/blob/master/Validation-ACZ_Based_TreeSuitblty/Validation_FieldOnly_AdjustedRasters.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Field-only validation of agroclimatic-zone-adjusted tree suitability rasters

This notebook uses the 20 original suitability assets as inputs, applies the agroclimatic-zone adjustment from `AC_zones_TreeSuitability.ipynb`, and validates the **adjusted images** at field occurrence points. It computes the adjusted images on the fly, so prior export of `adjusted_raster` assets is not required.

Before running, confirm that the zone field `AgClZn_Nam` and the zone names in `RECOMMENDED_ZONES` match the agroclimatic-zone asset.

In [ ]:
import os
import time
import ee
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# ==============================================================================
# 1. INITIALIZE EARTH ENGINE
# ==============================================================================
try:
    ee.Initialize(project='ee-geoinformers')
    print("Earth Engine initialized successfully.")
except Exception as e:
    print("Initializing EE with authentication prompt...")
    ee.Authenticate()
    ee.Initialize(project='ee-geoinformers')

# ==============================================================================
# 2. CONFIGURATION & ASSET MAPPINGS
# ==============================================================================
CSV_FILE_PATH = "1NicheMod_Combined_CLEANED_V2filtered_data.csv"
OUTPUT_EXCEL = "Species_Validation_Report_FieldOnly.xlsx"
OUTPUT_PLOT_DIR = "PE_Curves"
GRID_PLOT_FILE = "All_Species_Density_Distributions_FieldOnly.png"

os.makedirs(OUTPUT_PLOT_DIR, exist_ok=True)

# Column names in your CSV
LON_COL = "Longitude"
LAT_COL = "Latitude"
SPECIES_COL = "Botanical_Name"
SOURCE_COL = "Source"             # Column name for data source
SOURCE_FILTER_VAL = "Field"      # Only keep records matching this value

# Assets mapping for target species
RAW_SPECIES_ASSETS = {
    'Acacia nilotica': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Acacia_nilotica',
    'Ailanthus excelsa': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Ailanthus_excelsa',
    'Pongamia pinnata': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Pongamia_pinnata',
    'Populus deltoides': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Populus_deltoides',
    'Prosopis cineraria': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Prosopis_cineraria',
    'Terminalia arjuna': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Terminalia_arjuna',
    'Acacia mangium': 'projects/ee-geoinformers/assets/MoAFW/Regional_Suitability_All_Trees/Acacia_mangium',
    'Gliricidia sepium': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Gliricidia_sepium',
    'Anthocephalus cadamba': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Anthocephalus_cadamba',
    'Grewia optiva': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Grewia_optiva',
    'Mangifera indica': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Mangifera_indica',
    'Azadirachta indica': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Azadirachta_indica',
    'Albizia lebbeck': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Albizia_lebbeck',
    'Casuarina equisetifolia': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Casuarina_equisetifolia',
    'Dalbergia sissoo': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Dalbergia_sissoo',
    'Bambusa vulgaris': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Bambusa_vulgaris',
    'Eucalyptus spp.': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Eucalyptus_spp',
    'Gmelina arborea': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Gmelina_arborea',
    'Melia dubia': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Melia_dubia',
    'Tectona grandis': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Tectona_grandis'
}

# ==============================================================================
# 2B. AGROCLIMATIC-ZONE-ADJUSTED SUITABILITY
# ==============================================================================
# The adjusted image is computed on the fly; no raster export is required.
zone_fc = ee.FeatureCollection(
    "users/GEOINFORMERS/AgroClimaticZones/Agro-ClimateZone_WGS84"
)
REWARD_WEIGHT = 1.20
PENALTY_WEIGHT = 0.80

RECOMMENDED_ZONES = {
    'Acacia nilotica': ["Middle Gangetic Plains Region", "Upper Gangetic Plains Region", "Trans Gangetic Plains Region", "Central Plateau and Hill Region", "Western Plateau and Hill Region", "Southern Plateau and Hill Region", "East Coast Plains and Hill Region", "Gujrat Plains and Hill Region", "Western Dry Region"],
    'Ailanthus excelsa': ["Western Dry Region"],
    'Pongamia pinnata': ["Lower Gangetic Plains Region", "Eastern Plateau and Hill Region", "Central Plateau and Hill Region", "Western Plateau and Hill Region", "Southern Plateau and Hill Region", "East Coast Plains and Hill Region", "Gujrat Plains and Hill Region"],
    'Populus deltoides': ["Western Himalayan Region", "Middle Gangetic Plains Region", "Upper Gangetic Plains Region", "Trans Gangetic Plains Region"],
    'Prosopis cineraria': ["Trans Gangetic Plains Region", "Western Dry Region"],
    'Terminalia arjuna': ["Lower Gangetic Plains Region", "Middle Gangetic Plains Region", "Eastern Plateau and Hill Region"],
    'Acacia mangium': ["West Coast Plains and Hill Region"],
    'Gliricidia sepium': ["Southern Plateau and Hill Region", "West Coast Plains and Hill Region"],
    'Anthocephalus cadamba': ["Lower Gangetic Plains Region", "Middle Gangetic Plains Region", "Upper Gangetic Plains Region", "Trans Gangetic Plains Region"],
    'Grewia optiva': ["Western Himalayan Region"],
    'Mangifera indica': ["Lower Gangetic Plains Region", "Middle Gangetic Plains Region", "Upper Gangetic Plains Region"],
    'Azadirachta indica': ["Upper Gangetic Plains Region", "Western Plateau and Hill Region", "Southern Plateau and Hill Region", "East Coast Plains and Hill Region", "Gujrat Plains and Hill Region", "Western Dry Region"],
    'Albizia lebbeck': ["Trans Gangetic Plains Region"],
    'Casuarina equisetifolia': ["Lower Gangetic Plains Region", "Southern Plateau and Hill Region", "East Coast Plains and Hill Region", "Gujrat Plains and Hill Region", "The Island Region"],
    'Dalbergia sissoo': ["Middle Gangetic Plains Region", "Upper Gangetic Plains Region", "Trans Gangetic Plains Region", "Gujrat Plains and Hill Region"],
    'Bambusa vulgaris': ["Eastern Himalayan Region", "Lower Gangetic Plains Region", "West Coast Plains and Hill Region"],
    'Eucalyptus spp.': ["Lower Gangetic Plains Region", "Middle Gangetic Plains Region", "Upper Gangetic Plains Region", "Trans Gangetic Plains Region", "Eastern Plateau and Hill Region", "Central Plateau and Hill Region", "Western Plateau and Hill Region", "East Coast Plains and Hill Region", "Gujrat Plains and Hill Region"],
    'Gmelina arborea': ["Eastern Himalayan Region", "Lower Gangetic Plains Region", "Eastern Plateau and Hill Region"],
    'Melia dubia': ["Middle Gangetic Plains Region", "Upper Gangetic Plains Region", "Central Plateau and Hill Region", "Western Plateau and Hill Region", "Southern Plateau and Hill Region", "East Coast Plains and Hill Region", "West Coast Plains and Hill Region", "Gujrat Plains and Hill Region"],
    'Tectona grandis': ["Central Plateau and Hill Region", "Western Plateau and Hill Region", "Southern Plateau and Hill Region", "East Coast Plains and Hill Region", "West Coast Plains and Hill Region", "Gujrat Plains and Hill Region"]
}

def get_adjusted_suitability(species_name):
    """Build the zone-adjusted suitability image for a species (nominal scale 0-100)."""
    raw = ee.Image(RAW_SPECIES_ASSETS[species_name]).rename('original')
    rec_zones = RECOMMENDED_ZONES.get(species_name, [])
    base_zones = ee.Image(PENALTY_WEIGHT).paint(zone_fc, PENALTY_WEIGHT)
    rec_fc = zone_fc.filter(ee.Filter.inList('AgClZn_Nam', rec_zones))
    rec_painted = ee.Image().paint(rec_fc, REWARD_WEIGHT)
    multiplier = base_zones.where(rec_painted.mask(), REWARD_WEIGHT)
    kernel = ee.Kernel.gaussian(radius=15, units='pixels', sigma=3)
    smoothed_multiplier = multiplier.convolve(kernel)
    return raw.multiply(smoothed_multiplier).clamp(0, 100).rename('suitability')

# ==============================================================================
# 3. CONTINUOUS BOYCE INDEX ENGINE
# ==============================================================================
def calculate_continuous_boyce_ecospat(presence_vals, bg_vals, window_width=0.1, step=0.02):
    presence_vals = np.asarray(presence_vals, dtype=float)
    bg_vals = np.asarray(bg_vals, dtype=float)

    presence_vals = presence_vals[np.isfinite(presence_vals)]
    presence_vals = presence_vals[(presence_vals >= 0.0) & (presence_vals <= 1.0)]
    bg_vals = bg_vals[np.isfinite(bg_vals)]
    bg_vals = bg_vals[(bg_vals >= 0.0) & (bg_vals <= 1.0)]

    if len(presence_vals) < 5 or len(bg_vals) < 100:
        return np.nan

    presence_vals = np.sort(np.unique(presence_vals))
    if len(presence_vals) < 5:
        return np.nan

    total_p = len(presence_vals)
    total_e = len(bg_vals)

    window_starts = np.arange(0, 1.0 - window_width + 1e-5, step)
    pe_ratios, hs_midpoints = [], []

    for w_start in window_starts:
        w_end = w_start + window_width
        w_mid = (w_start + w_end) / 2.0
        p_i = np.sum((presence_vals >= w_start) & (presence_vals <= w_end))
        e_i = np.sum((bg_vals >= w_start) & (bg_vals <= w_end))

        if e_i > 0:
            pe_ratios.append((p_i / total_p) / (e_i / total_e))
            hs_midpoints.append(w_mid)

    pe_ratios, hs_midpoints = np.array(pe_ratios), np.array(hs_midpoints)
    if len(pe_ratios) < 5:
        return np.nan

    if len(pe_ratios) >= 3:
        pe_smoothed = np.convolve(pe_ratios, np.ones(3) / 3.0, mode='valid')
        hs_midpoints = hs_midpoints[1:-1]
    else:
        pe_smoothed = pe_ratios

    _, unique_indices = np.unique(np.round(pe_smoothed, 6), return_index=True)
    pe_final = pe_smoothed[np.sort(unique_indices)]
    hs_final = hs_midpoints[np.sort(unique_indices)]

    if len(pe_final) < 5:
        return np.nan

    rho, _ = stats.spearmanr(hs_final, pe_final)
    return rho

def derive_validation_status(boyce, n_points, mean_suit):
    if n_points < 10:
        return "Insufficient observations"
    if pd.isna(boyce):
        return "Unrated"
    if boyce >= 0.75:
        return "Good agreement"
    elif boyce >= 0.50:
        if mean_suit >= 0.60:
            return "Good agreement"
        return "Moderate agreement"
    elif boyce >= 0.30:
        return "Weak agreement"
    else:
        return "Poor agreement"

def derive_confidence(n_points):
    if n_points >= 100:
        return "High"
    elif n_points >= 30:
        return "Medium"
    else:
        return "Low"

# ==============================================================================
# 4. FIXED EXECUTION PIPELINE (With Diagnostic Tracking)
# ==============================================================================
def execute_sdm_validation():
    df_occurrences = pd.read_csv(CSV_FILE_PATH)
    df_occurrences.columns = df_occurrences.columns.str.strip()

    print("=== DATASET DIAGNOSTICS BEFORE FILTERING ===")
    print(f"Total rows in CSV: {len(df_occurrences)}")
    if SOURCE_COL in df_occurrences.columns:
        print(f"Unique values in '{SOURCE_COL}' column: {df_occurrences[SOURCE_COL].unique().tolist()}")

        # Robust filtering: handles 'Field', 'field', 'Field Survey', 'Field Data' and strips trailing spaces
        df_occurrences = df_occurrences[
            df_occurrences[SOURCE_COL].astype(str).str.strip().str.contains(SOURCE_FILTER_VAL, case=False, na=False)
        ].copy()
        print(f"Total rows kept after '{SOURCE_FILTER_VAL}' filter: {len(df_occurrences)}\n")
    else:
        print(f"⚠️ Warning: Column '{SOURCE_COL}' not found in CSV. Using all rows.\n")

    report_rows = []
    species_density_data = {}

    species_list = list(RAW_SPECIES_ASSETS.keys())
    total_species = len(species_list)

    for sp_idx, species_name in enumerate(species_list, start=1):
        # Case-insensitive & stripped string matching for species names
        sp_df = df_occurrences[
            df_occurrences[SPECIES_COL].astype(str).str.strip().str.lower() == species_name.strip().lower()
        ].drop_duplicates(subset=[LON_COL, LAT_COL])

        total_pts = len(sp_df)
        print(f"[{sp_idx}/{total_species}] {species_name} --> Found {total_pts} unique '{SOURCE_FILTER_VAL}' points")

        if total_pts == 0:
            print(f"  ❌ Skipped: 0 field points found for {species_name}.")
            status = "Insufficient observations"
            confidence = "Low"
            report_rows.append({
                'Species': species_name,
                'Total Points': 0,
                'Confidence': confidence,
                'Valid Points': 0,
                'Mean ± SD': "N/A",
                'Mean Suitability': np.nan,
                'Median': np.nan,
                'SD': np.nan,
                'Q1': np.nan,
                'Q3': np.nan,
                '90th percentile': np.nan,
                'Boyce': np.nan,
                'High (%)': "N/A",
                'Moderate (%)': "N/A",
                'Low (%)': "N/A",
                'Validation Status': status
            })
            continue

        features = [
            ee.Feature(ee.Geometry.Point([float(r[LON_COL]), float(r[LAT_COL])]))
            for _, r in sp_df.iterrows()
        ]
        fc = ee.FeatureCollection(features)
        suitability_img = get_adjusted_suitability(species_name)
        suitability_img = suitability_img.updateMask(suitability_img.gte(0))

        # Sample Presence Points
        try:
            sampled_fc = suitability_img.sampleRegions(collection=fc, scale=30, geometries=False).filter(ee.Filter.notNull(['suitability']))
            extracted_vals = sampled_fc.aggregate_array('suitability').getInfo()
            presence_vals = np.array([v / 100.0 if v > 1.0 else float(v) for v in extracted_vals if v is not None and v >= 0])
            print(f"  ✓ GEE overlay successfully extracted {len(presence_vals)} valid pixel values.")
        except Exception as e:
            print(f"  ❌ GEE Sampling Error: {e}")
            presence_vals = np.array([])

        # Sample Background Points
        try:
            bg_fc = suitability_img.sample(numPixels=25000, scale=1000, dropNulls=True, geometries=False)
            bg_raw = bg_fc.aggregate_array('suitability').getInfo()
            bg_vals = np.array([v / 100.0 if v > 1.0 else float(v) for v in bg_raw if v is not None and v >= 0])
        except Exception:
            bg_vals = np.linspace(0.0, 1.0, 10000)

        valid_pts = len(presence_vals)
        if valid_pts > 0:
            mean_val = np.mean(presence_vals)
            sd_val = np.std(presence_vals)
            median_val = np.median(presence_vals)
            q1_val = np.percentile(presence_vals, 25)
            q3_val = np.percentile(presence_vals, 75)
            p90_val = np.percentile(presence_vals, 90)

            high_pct = np.mean(presence_vals >= 0.70) * 100
            mod_pct = np.mean((presence_vals >= 0.40) & (presence_vals < 0.70)) * 100
            low_pct = np.mean(presence_vals < 0.40) * 100
        else:
            mean_val = sd_val = median_val = q1_val = q3_val = p90_val = np.nan
            high_pct = mod_pct = low_pct = np.nan

        boyce_val = calculate_continuous_boyce_ecospat(presence_vals, bg_vals)
        status = derive_validation_status(boyce_val, valid_pts, mean_val)
        confidence = derive_confidence(total_pts)

        species_density_data[species_name] = {
            'presence_vals': presence_vals,
            'mean': mean_val,
            'p90': p90_val,
            'status': status,
            'boyce': boyce_val
        }

        report_rows.append({
            'Species': species_name,
            'Total Points': total_pts,
            'Confidence': confidence,
            'Valid Points': valid_pts,
            'Mean ± SD': f"{mean_val:.2f} ± {sd_val:.2f}" if pd.notna(mean_val) else "N/A",
            'Mean Suitability': mean_val,
            'Median': median_val,
            'SD': sd_val,
            'Q1': q1_val,
            'Q3': q3_val,
            '90th percentile': p90_val,
            'Boyce': boyce_val,
            'High (%)': f"{high_pct:.1f}%" if pd.notna(high_pct) else "N/A",
            'Moderate (%)': f"{mod_pct:.1f}%" if pd.notna(mod_pct) else "N/A",
            'Low (%)': f"{low_pct:.1f}%" if pd.notna(low_pct) else "N/A",
            'Validation Status': status
        })

        time.sleep(1)

    # Export Excel Summary Report
    df_report = pd.DataFrame(report_rows)
    with pd.ExcelWriter(OUTPUT_EXCEL, engine='openpyxl') as writer:
        df_report.to_excel(writer, sheet_name='Validation_Status', index=False)

    print(f"\n✅ Saved Excel report: {OUTPUT_EXCEL}")

    # Generate Grid Density Image
    fig, axes = plt.subplots(4, 5, figsize=(20, 15), sharex=True, sharey=True)
    axes = axes.flatten()

    for idx, species_name in enumerate(species_list):
        ax = axes[idx]
        data = species_density_data.get(species_name, None)

        if data is not None and len(data['presence_vals']) >= 5:
            vals = data['presence_vals']
            density = stats.gaussian_kde(vals)
            xs = np.linspace(0, 1, 200)
            ys = density(xs)

            ax.plot(xs, ys, color='darkgreen', linewidth=1.5, label='Density Curve')
            ax.fill_between(xs, 0, ys, color='forestgreen', alpha=0.25)

            if pd.notna(data['p90']):
                ax.axvline(data['p90'], color='crimson', linestyle='--', label=f"90th Pct ({data['p90']:.2f})")
            if pd.notna(data['mean']):
                ax.axvline(data['mean'], color='steelblue', linestyle=':', label=f"Mean ({data['mean']:.2f})")

            ax.set_title(f"{species_name}\nStatus: {data['status']}", fontsize=10, fontweight='bold')
            ax.legend(loc='upper left', fontsize=7)
        else:
            ax.set_title(f"{species_name}\nStatus: Insufficient observations", fontsize=10, fontweight='bold')

        ax.set_xlim(0.0, 1.0)
        ax.grid(True, linestyle=':', alpha=0.5)
        ax.set_xlabel("Suitability", fontsize=8)
        ax.set_ylabel("Density", fontsize=8)

    for idx in range(len(species_list), len(axes)):
        fig.delaxes(axes[idx])

    plt.tight_layout()
    plt.savefig(GRID_PLOT_FILE, dpi=300)
    plt.close()
    print(f"✅ Saved Combined Density Grid: {GRID_PLOT_FILE}")

execute_sdm_validation()

Earth Engine initialized successfully.
=== DATASET DIAGNOSTICS BEFORE FILTERING ===
Total rows in CSV: 105850
Unique values in 'Source' column: ['Field', 'GBIF']
Total rows kept after 'Field' filter: 3850

[1/20] Acacia nilotica --> Found 438 unique 'Field' points
  ✓ GEE overlay successfully extracted 438 valid pixel values.
[2/20] Ailanthus excelsa --> Found 41 unique 'Field' points
  ✓ GEE overlay successfully extracted 41 valid pixel values.
[3/20] Pongamia pinnata --> Found 45 unique 'Field' points
  ✓ GEE overlay successfully extracted 45 valid pixel values.
[4/20] Populus deltoides --> Found 2 unique 'Field' points
  ✓ GEE overlay successfully extracted 2 valid pixel values.
[5/20] Prosopis cineraria --> Found 24 unique 'Field' points
  ✓ GEE overlay successfully extracted 24 valid pixel values.
[6/20] Terminalia arjuna --> Found 29 unique 'Field' points
  ✓ GEE overlay successfully extracted 29 valid pixel values.
[7/20] Acacia mangium --> Found 0 unique 'Field' points
  ❌ Skip

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ==============================================================================
# VISUALIZATION
# ==============================================================================
import geemap

Map = geemap.Map(center=[22.5937, 78.9629], zoom=5, basemap='HYBRID')

species_to_test = 'Gmelina arborea'
original_img, adjusted_img = get_adjusted_suitability(species_to_test)

vis_params = {
    'min': 0,
    'max': 0.8,
    'palette': ['#d73027', '#f46d43', '#fdae61', '#fee08b', '#d9ef8b', '#a6d96a', '#1a9850']
}

Map.addLayer(original_img, vis_params, 'Original')
Map.addLayer(adjusted_img, vis_params, 'Adjusted')

Map.add_layer_control()
Map